In [0]:
%run "./Util"

In [0]:
# ── CELL 1 : Imports + read silver ───────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType

silver = spark.sql(f" SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")


In [0]:
def get_merge_stats(DIM_TABLE):
    return spark.sql(f"DESCRIBE HISTORY {CATALOG}.{GOLD_SCHEMA}.{DIM_TABLE} ") \
        .select("*") \
        .orderBy(F.col("version").desc()) \
        .limit(1) \
        .withColumn("Status",
        F.when(F.col("operationMetrics.numSourceRows") == 0, "No Data to Load")
        .when(F.col("operationMetrics.numTargetRowsInserted") != 0, "Data Inserted")
        .when(
            (F.col("operationMetrics.numSourceRows") != 0) & (F.col("operationMetrics.numTargetRowsInserted") == 0),
            "No Change in Data"
        ).otherwise("Failure"))\
        .selectExpr("timestamp", "job", "operation", "operationMetrics.numOutputRows", "operationMetrics.numSourceRows", "operationMetrics.numTargetRowsInserted", "Status")


def quarantine_fact_inspection(quarantine_df):
    quarantine_df.createOrReplaceTempView("staging_quarantine_inspection")
    spark.sql(f""" 
        INSERT INTO {CATALOG}.{GOLD_SCHEMA}.{GOLD_QUARANTINE_INSPECTION}
        SELECT
            inspection_id, 
            restaurant_id,
            restaurant_name, 
            aka_name,  
            CAST(inspection_date AS DATE), 
            inspection_type, 
            inspection_result,
            violation_score,
            violation_code,
            violation_points,
            pass_fail_flag, 
            source_city, 
            address, 
            zip_code,
            quarantine_category,
            failed_column,
            source_system,
            pipeline_name,
            current_timestamp() AS _date_to_warehouse,
            current_timestamp()AS quarantine_timestamp
        FROM staging_quarantine_inspection
    """)

## Load Dim Date

In [0]:
# ── dim_date ────────────────────────────────────────

dim_date = (
    silver
    .select("inspection_date")
    .distinct()
    .filter(F.col("inspection_date").isNotNull())
    .withColumn("date_key",           F.date_format("inspection_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("full_date",          F.col("inspection_date"))
    .withColumn("year",               F.year("inspection_date"))
    .withColumn("month_name",         F.date_format("inspection_date", "MMMM"))
    .withColumn("month_number",       F.month("inspection_date"))
    .withColumn("day",                F.date_format("inspection_date", "EEEE"))
    .withColumn("quarter",            F.concat(F.lit("Q"), F.quarter("inspection_date")))
    .withColumn("week_number",        F.weekofyear("inspection_date"))
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "date_key", "full_date", "year", "month_name",
        "month_number", "day", "quarter", "week_number",
        "_date_to_warehouse"
    )
)

dim_date.createOrReplaceTempView("staging_dim_date")

spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_DATE} AS target
    USING staging_dim_date AS source
    ON target.date_key = source.date_key
    WHEN NOT MATCHED THEN INSERT *
""")

# dim_date.write.format("delta").mode("append")\
#     .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_DATE}")

get_merge_stats(DIM_DATE).display()


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-11T23:32:45.000Z,null,MERGE,1,2289,1,Data Inserted


## Load Dim Violation Detail

In [0]:
# ── dim_violation_detail ───────────────────────────────────

dim_violation = (
    silver
    .select("violation_id", "violation_code", "violation_description", "violation_detail")
    .distinct()
    .filter(
        F.col("violation_code").isNotNull() &
        (F.trim(F.col("violation_code")) != ""))
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "violation_id", "violation_code",
        "violation_description", "violation_detail", "_date_to_warehouse"
    )
)

dim_violation.createOrReplaceTempView("staging_dim_violation_detail")

spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL} AS target
    USING staging_dim_violation_detail AS source
    ON target.violation_id = source.violation_id
    WHEN NOT MATCHED THEN INSERT (
        violation_id, violation_code, violation_description, violation_detail, _date_to_warehouse
    )
    VALUES (
        source.violation_id, source.violation_code, source.violation_description, source.violation_detail, source._date_to_warehouse
    )
""")


get_merge_stats(DIM_VIOLATION_DETAIL).display()

timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-11T23:32:50.000Z,null,MERGE,0,739,0,No Change in Data


## Load Dim Restaurant

In [0]:
# dim_restaurant = (
#     silver
#     .select("inspection_date", "restaurant_id", "restaurant_name", "aka_name", "risk_category", "facility_type", "address", "source_city", "state", "zip_code", "change_hash")
#     .distinct()
#     .orderBy("restaurant_id", "inspection_date")
# )

# w = Window.partitionBy("restaurant_id").orderBy("inspection_date")
# # Addd Sequence number for each restaurant_id records in case multiple records available
# df_seq = dim_restaurant.withColumn("seq", F.row_number().over(w))
# max_seq = df_seq.agg(F.max("seq")).collect()[0][0]
# for i in range(1, max_seq + 1):

#     batch_i = df_seq.filter(F.col("seq") == i)
#     batch_i.createOrReplaceTempView("batch_i")

#     spark.sql(f"""
#         MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT} target
#         USING batch_i source
#         ON target.restaurant_id = source.restaurant_id 
#         AND target.is_current = true

#         -- Expire old record if change
#         WHEN MATCHED AND target.change_hash <> source.change_hash THEN
#         UPDATE SET
#             target.effective_end_date = source.inspection_date,
#             target.is_current = false

#         -- Insert new record
#         WHEN NOT MATCHED THEN
#         INSERT (
#             restaurant_id, dba_name, aka_name, risk_category, facility_type, address, city, state, zip_code, effective_start_date, effective_end_date, is_current, change_hash, _date_to_warehouse
#         )
#         VALUES (
#             source.restaurant_id, source.restaurant_name, source.aka_name, source.risk_category, source.facility_type, source.address, source.source_city, source.state, source.zip_code, source.inspection_date, TIMESTAMP '9999-12-31', true,
#             source.change_hash, current_timestamp()
#         )
#     """)

# # dim_restaurant.createOrReplaceTempView("staging_dim_restaurant")
# get_merge_stats(DIM_RESTAURANT).display()

In [0]:
# ── dim_restaurant ───────────────────────────────────

dim_restaurant = (
    silver
    .select("restaurant_id", "dba_name", "aka_name", "facility_type", "address", "source_city", "state", "zip_code", "change_hash")
    .distinct()
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .alias("s")
    .join(
        spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT}")
        .filter("is_current = true")
        .select("restaurant_id", "change_hash")
        .alias("t"),
        on=[
            F.col("s.restaurant_id") == F.col("t.restaurant_id"),
            F.col("s.change_hash") == F.col("t.change_hash")
        ],
        how="left_anti"
    )
)
dim_restaurant.createOrReplaceTempView("staging_dim_restaurant")


In [0]:
%sql

-- TRUNCATE TABLE Final_Project.Gold_Zone.dim_restaurant

In [0]:
%sql

-- WITH dedup_source AS (
--   SELECT *
--   FROM (
--     SELECT *,
--            ROW_NUMBER() OVER (
--              PARTITION BY restaurant_id
--              ORDER BY _date_to_warehouse DESC
--            ) AS rn
--     FROM staging_dim_restaurant
--   )
--   WHERE rn = 1
-- )

MERGE INTO final_project.gold_zone.dim_restaurant AS target
USING (
  SELECT restaurant_id AS merge_key, * FROM staging_dim_restaurant
  UNION ALL
  SELECT NULL AS merge_key, * FROM staging_dim_restaurant
) AS source
ON target.restaurant_id = source.merge_key
AND target.is_current = true

-- Expire old
WHEN MATCHED AND target.change_hash <> source.change_hash THEN
  UPDATE SET
    target.is_current = false,
    target.effective_end_date = current_timestamp()

-- Insert ONLY from NULL branch
WHEN NOT MATCHED AND source.merge_key IS NULL THEN
  INSERT (
    restaurant_id,
    dba_name,
    aka_name,
    facility_type,
    address,
    city,
    state,
    zip_code,
    change_hash,
    effective_start_date,
    effective_end_date,
    is_current,
    _date_to_warehouse
  )
  VALUES (
    source.restaurant_id,
    source.dba_name,
    source.aka_name,
    source.facility_type,
    source.address,
    source.source_city,
    source.state,
    source.zip_code,
    source.change_hash,
    current_timestamp(),
    TIMESTAMP '9999-12-31',
    true,
    source._date_to_warehouse
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql

-- select restaurant_id, count(distinct restaurant_key) 
-- from final_project.gold_zone.dim_restaurant
-- where is_current = true
-- group by 1
-- having count(distinct restaurant_key) = 1
-- order by 2 desc

restaurant_key,restaurant_id,dba_name,aka_name,facility_type,address,city,state,zip_code,change_hash,effective_start_date,effective_end_date,is_current,_date_to_warehouse
279034,3fafd6a11680a5cd09e9b8f577e1c2e8,BRENDA'S KIDS CLUB DAYCARE CENTER,BRENDA'S KIDS CLUB DAYCARE CENTER SANKET,DAYCARE ABOVE AND UNDER 2 YEARS,3552 E 118TH ST,CHICAGO,IL,60617,cc77996e226560e87b96a916912b5096,2026-04-11T00:00:00.000Z,2026-04-11T00:00:00.000Z,false,2026-04-11T22:56:12.512Z
279038,3fafd6a11680a5cd09e9b8f577e1c2e8,BRENDA'S KIDS CLUB DAYCARE CENTER,BRENDA'S KIDS CLUB DAYCARE CENTER SANKET,DAYCARE ABOVE AND UNDER 2 YEARS,3552 E 118TH ST,CHICAGO,IL,60617,cc77996e226560e87b96a916912b5096,2026-04-11T23:20:59.293Z,2026-04-11T23:25:16.499Z,false,2026-04-11T23:20:59.293Z
279042,3fafd6a11680a5cd09e9b8f577e1c2e8,BRENDA'S KIDS CLUB DAYCARE CENTER,BRENDA'S KIDS CLUB DAYCARE CENTER KOMAL,DAYCARE ABOVE AND UNDER 2 YEARS,3552 E 118TH ST,CHICAGO,IL,60617,4581d94414b25cd5d250c3fd3aeeb9ac,2026-04-11T23:32:54.338Z,9999-12-31T00:00:00.000Z,true,2026-04-11T23:32:54.338Z
279036,3fafd6a11680a5cd09e9b8f577e1c2e8,BRENDA'S KIDS CLUB DAYCARE CENTER,BRENDA'S KIDS CLUB DAYCARE CENTER,DAYCARE ABOVE AND UNDER 2 YEARS,3552 E 118TH ST,CHICAGO,IL,60617,39c93bf5622d0c0f5e6b955459a99c77,2026-04-11T00:00:00.000Z,2026-04-11T23:20:59.293Z,false,2026-04-11T23:04:58.815Z
246536,3fafd6a11680a5cd09e9b8f577e1c2e8,BRENDA'S KIDS CLUB DAYCARE CENTER,BRENDA'S KIDS CLUB DAYCARE CENTER,DAYCARE ABOVE AND UNDER 2 YEARS,3552 E 118TH ST,CHICAGO,IL,60617,39c93bf5622d0c0f5e6b955459a99c77,2026-04-11T00:00:00.000Z,2026-04-11T00:00:00.000Z,false,2026-04-11T21:57:29.913Z
279040,3fafd6a11680a5cd09e9b8f577e1c2e8,BRENDA'S KIDS CLUB DAYCARE CENTER,BRENDA'S KIDS CLUB DAYCARE CENTER,DAYCARE ABOVE AND UNDER 2 YEARS,3552 E 118TH ST,CHICAGO,IL,60617,39c93bf5622d0c0f5e6b955459a99c77,2026-04-11T23:25:16.499Z,2026-04-11T23:32:54.338Z,false,2026-04-11T23:25:16.499Z


In [0]:
# %sql

# select * from final_project.bronze_zone.bronze_chicago
# where dba_name = "BRENDA'S KIDS CLUB DAYCARE CENTER"

In [0]:
# %sql
# Working COde
# WITH base_dedup AS (
#     SELECT *
#     FROM (
#         SELECT *,
#                ROW_NUMBER() OVER (
#                    PARTITION BY restaurant_id, risk_category, facility_type
#                    ORDER BY inspection_date
#                ) as rn
#         FROM staging_dim_restaurant
#     ) 
#     -- WHERE rn = 1
# ),

# ordered AS (
#     SELECT *,
#         LAG(change_hash) OVER (
#             PARTITION BY restaurant_id, risk_category, facility_type 
#             ORDER BY inspection_date
#         ) AS prev_hash
#     FROM base_dedup
# ),

# -- changess AS (
# --     SELECT * FROM ordered 
# --     WHERE prev_hash IS NULL OR prev_hash <> change_hash
# -- ),

# -- scd_rows AS (
# --     SELECT  *,
# --         inspection_date AS effective_start_date,
# --         LEAD(inspection_date) OVER ( PARTITION BY restaurant_id, risk_category, facility_type  ORDER BY inspection_date) AS next_date
# --     FROM changess
# -- ),

# ordered2 AS (
#     SELECT *,
#         CASE 
#             WHEN LAG(change_hash) OVER (PARTITION BY restaurant_id, risk_category, facility_type ORDER BY inspection_date) 
#                  = change_hash 
#             THEN 0 ELSE 1 
#         END AS change_flag
#     FROM ordered
# ),

# grp AS (
#     SELECT *,
#         SUM(change_flag) OVER (
#             PARTITION BY restaurant_id, facility_type 
#             ORDER BY inspection_date
#         ) AS grp_id
#     FROM ordered2
# ),

# scd_rows2 AS (
#     SELECT
#         restaurant_id,
#         restaurant_name,
#         aka_name,
#         risk_category,
#         facility_type,
#         address,
#         source_city,
#         state,
#         zip_code,
#         change_hash,

#         MIN(inspection_date) AS effective_start_date,
#         MAX(inspection_date) AS effective_end_date

#     FROM grp
#     GROUP BY
#         restaurant_id,
#         restaurant_name,
#         aka_name,
#         risk_category,
#         facility_type,
#         address,
#         source_city,
#         state,
#         zip_code,
#         change_hash,
#         grp_id
# )

# MERGE INTO Final_Project.Gold_Zone.dim_restaurant target
# USING scd_rows2 source

# ON target.restaurant_id = source.restaurant_id
# AND target.effective_start_date = source.effective_start_date

# -- Only insert new SCD rows
# WHEN NOT MATCHED THEN
# INSERT (
#     restaurant_id,
#     dba_name,
#     aka_name,
#     risk_category,
#     facility_type,
#     address,
#     city,
#     state,
#     zip_code,
#     effective_start_date,
#     effective_end_date,
#     is_current,
#     change_hash,
#     _date_to_warehouse
# )
# VALUES (
#     source.restaurant_id,
#     source.restaurant_name,
#     source.aka_name,
#     source.risk_category,
#     source.facility_type,
#     source.address,
#     source.source_city,
#     source.state,
#     source.zip_code,
#     source.effective_start_date,
#     COALESCE(source.effective_end_date, TIMESTAMP '9999-12-31'),
#     CASE WHEN source.effective_end_date IS NULL THEN true ELSE false END,
#     source.change_hash,
#     current_timestamp()
# )


In [0]:
# %sql

# select * from final_project.gold_zone.dim_restaurant

# -- group by 1
# where restaurant_id = "dd3c467d63a19102edb56bcc793dd6d8"

In [0]:
# %sql
# WITH base_dedup AS (
#     SELECT *
#     FROM (
#         SELECT *,
#                ROW_NUMBER() OVER (
#                    PARTITION BY restaurant_id, risk_category, facility_type
#                    ORDER BY inspection_date
#                ) as rn
#         FROM staging_dim_restaurant
#     ) 
#     -- WHERE rn = 1
# ),

# ordered AS (
#     SELECT *,
#         LAG(change_hash) OVER (
#             PARTITION BY restaurant_id, risk_category, facility_type
#             ORDER BY inspection_date
#         ) AS prev_hash,
#         LEAD(change_hash) OVER (
#             PARTITION BY restaurant_id, risk_category, facility_type
#             ORDER BY inspection_date
#         ) AS next_hash
#     FROM base_dedup
# ),

# changess AS (
#     SELECT * FROM ordered 
#     WHERE prev_hash IS NULL OR prev_hash <> change_hash OR next_hash IS NULL
# ),

# scd_rows AS (
#     SELECT  *,
#         inspection_date AS effective_start_date,
#         CASE 
#         WHEN LEAD(inspection_date) OVER ( PARTITION BY restaurant_id, risk_category, facility_type  ORDER BY inspection_date ) IS NULL THEN inspection_date
#         ELSE LEAD(inspection_date) OVER ( PARTITION BY restaurant_id, risk_category, facility_type  ORDER BY inspection_date)
#         END AS next_date
#     FROM changess
# ),

# ordered2 AS (
#     SELECT *,
#         CASE 
#             WHEN LAG(change_hash) OVER (PARTITION BY restaurant_id, facility_type ORDER BY inspection_date) 
#                  = change_hash 
#             THEN 0 ELSE 1 
#         END AS change_flag
#     FROM ordered
# ),

# grp AS (
#     SELECT *,
#         SUM(change_flag) OVER (
#             PARTITION BY restaurant_id, facility_type 
#             ORDER BY inspection_date
#         ) AS grp_id
#     FROM ordered2
# ),

# scd_rows2 AS (
#     SELECT
#         restaurant_id,
#         restaurant_name,
#         aka_name,
#         risk_category,
#         facility_type,
#         address,
#         source_city,
#         state,
#         zip_code,
#         change_hash,

#         MIN(inspection_date) AS effective_start_date,
#         MAX(inspection_date) AS effective_end_date

#     FROM grp
#     GROUP BY
#         restaurant_id,
#         restaurant_name,
#         aka_name,
#         risk_category,
#         facility_type,
#         address,
#         source_city,
#         state,
#         zip_code,
#         change_hash,
#         grp_id
# )

# select * from scd_rows2
# -- where restaurant_id = "dd3c467d63a19102edb56bcc793dd6d8"
# where restaurant_id = "dd3c467d63a19102edb56bcc793dd6d8"


In [0]:
# %sql

# select distinct restaurant_id, restaurant_name, aka_name, facility_type, address, zip_code, change_hash, inspection_date
# from final_project.silver_zone.silver_table
# where restaurant_id = "1ee4f88c4bc0ae88df679852ec839eb7"

## Quarantine Duplicate Inspection if any

In [0]:
# quarantine_df = (silver_df.alias("s").join(
#     spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS}").alias("t"),
#     F.col("s.inspection_id") == F.col("t.inspection_id"),
#     "inner"
#     ).select("s.*")
#     .withColumn("quarantine_category", F.lit("Duplicates"))
#     .withColumn("failed_column", F.lit("inspection_id"))
#     .withColumn("source_system", F.lit("silver"))
#     .withColumn("pipeline_name", F.lit("gold"))
# )
# quarantine_fact_inspection(quarantine_df)

## Load Fact Inspection

In [0]:
%sql

-- TRUNCATE TABLE final_project.gold_zone.FACT_INSPECTIONS

In [0]:
# silver_df_agg = (
#     silver
#     .groupBy("inspection_id", "inspection_date", "restaurant_id")
#     .agg(F.countDistinct("violation_code").alias("total_violation"),
#         F.sum(F.coalesce(F.col("violation_points").cast(IntegerType()), F.lit(0))).alias("total_violation_points")
#     )
# ).alias("agg")

# silver_df_agg = (
#     silver_df_agg
#     .join(silver.alias("s"), on=["inspection_id", "inspection_date", "restaurant_id"], how="left")
#     .select("s.inspection_id", "s.restaurant_id", "restaurant_name", "aka_name", "source_city", "zip_code",
#         "s.inspection_date", "inspection_type", "risk_category", "facility_type", "address",
#         "zip_code", "violation_score", "inspection_result", "agg.total_violation", "agg.total_violation_points"
#     ).distinct()
# )

# silver_df_agg.filter(F.col("inspection_id") == "fefdb04655732f3fa053913afc3dbd61").display()

In [0]:
# ── Fact Inspection ───────────────────────────────────

dim_date_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_DATE}")
    .select("date_key", "full_date")
)

w = Window.partitionBy("restaurant_id").orderBy(F.col("restaurant_key").desc())

dim_rest_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT}")
    # .filter(F.col("is_current") == True)
    .select("restaurant_key", "restaurant_id", "dba_name", "aka_name", "facility_type", "change_hash")
    .distinct()
    # .withColumn("rn", F.row_number().over(w))
    # .filter(F.col("rn") == 1)
    # .drop("rn")
)

silver_df = silver.alias("s")
dim_rest_lkp = dim_rest_lkp.alias("r")
dim_date_lkp = dim_date_lkp.alias("dt")


silver_df_agg = (
    silver
    .groupBy("inspection_id", "inspection_date", "restaurant_id")
    .agg(F.countDistinct("violation_id").alias("total_violation").cast(IntegerType()),
        F.sum(F.coalesce(F.col("violation_points").cast(IntegerType()), F.lit(0))).alias("total_violation_points")
    )
).alias("agg")

fact_inspections = (
    silver_df_agg
    .join(silver.alias("s"), on=["inspection_id", "inspection_date", "restaurant_id"], how="left")
    .select("s.inspection_id", "s.restaurant_id", "dba_name", "aka_name", "source_city", "zip_code",
        "s.inspection_date", "inspection_type", "risk_category", "facility_type", "address", "license_num",
        "zip_code", "violation_score", "inspection_result", "pass_fail_flag", "agg.total_violation", "agg.total_violation_points"
    )
    .distinct()

    # Resolve date key
    .join(dim_date_lkp, F.col("s.inspection_date") == F.col("dt.full_date"), "left")

    # Resolve restaurant key
    .join(dim_rest_lkp,
          ((F.col("s.restaurant_id")    == F.col("r.restaurant_id"))   &
            (F.col("s.dba_name")        == F.col("r.dba_name"))        &
            (F.col("s.aka_name")        == F.col("r.aka_name"))
          ), "left")
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "inspection_id", "restaurant_key", F.col("license_num").cast(IntegerType()), "risk_category", F.col("date_key").alias("inspection_date_key"), F.col("violation_score").alias("inspection_score"), "inspection_result", 
        F.col("total_violation").cast(IntegerType()), F.col("total_violation_points").cast(IntegerType()), 
        "inspection_type", "source_city", "pass_fail_flag", "_date_to_warehouse"
    )
)

fact_inspections.createOrReplaceTempView("staging_fact_inspections")

# Append Only
spark.sql(f""" 
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS} target
    USING staging_fact_inspections source
    ON target.inspection_id = source.inspection_id
    -- Safe Append 
    WHEN NOT MATCHED THEN INSERT ( 
        inspection_id, restaurant_key, license_num, risk_category, inspection_date_key, inspection_score, inspection_result, total_violation, total_violation_points, inspection_type, source_city, pass_fail_flag, _date_to_warehouse
    )
    Values (
        source.inspection_id, source.restaurant_key, source.license_num, source.risk_category, source.inspection_date_key, source.inspection_score, source.inspection_result, source.total_violation, source.total_violation_points, source.inspection_type, source.source_city, source.pass_fail_flag, source._date_to_warehouse
    )
""")

get_merge_stats(FACT_INSPECTIONS).display()


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-11T23:33:13.000Z,null,MERGE,1,57739,1,Data Inserted


In [0]:
# %sql

# select Inspection_Key, count(1)
# from final_project.gold_zone.fact_inspections
# group by 1
# having count(1) > 1
# order by 2 desc

Inspection_Key,count(1)


## Load Fact Violations

In [0]:
%sql

-- SELECT inspection_id, COUNT(distinct inspection_key) 
-- FROM final_project.gold_zone.fact_inspections
-- -- where source_city = "CHICAGO"
-- GROUP BY 1
-- HAVING COUNT(distinct inspection_key)  > 1
-- ORDER BY 2 DESC

-- select * from final_project.gold_zone.fact_inspections
-- -- where source_city = "DALLAS"
-- where inspection_id = "1dd9df182e946538ee59d74e139c43f5"


-- 48237 duplicate inspection_id

In [0]:
# %sql

# select * from final_project.gold_zone.dim_restaurant
# where restaurant_id = "1ee4f88c4bc0ae88df679852ec839eb7"

In [0]:
# %sql

# select inspection_id, restaurant_id, restaurant_name, aka_name, source_city, zip_code,
#         inspection_date, inspection_type, inspection_result, risk_category, facility_type, address,
#         violation_points, violation_detail, violation_score, pass_fail_flag  
# from final_project.silver_zone.silver_table
# where inspection_id = "c69c05cdea027da85ea95c055a335054"

In [0]:
# %sql
# select inspection_id, restaurant_id, restaurant_name, aka_name, source_city, zip_code,
#         inspection_date, inspection_type, inspection_result, risk_category, facility_type, address,
#         violation_code, violation_points, violation_detail, violation_score, pass_fail_flag 
# from final_project.silver_zone.silver_table
# where inspection_id = "fefdb04655732f3fa053913afc3dbd61"

In [0]:
%sql

-- TRUNCATE TABLE final_project.gold_zone.fact_violations

In [0]:

dim_viol_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL}")
    .select("violation_code_key", "violation_id")
)

w = Window.partitionBy("inspection_id").orderBy(F.col("inspection_score").desc())

fact_insp_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS}")
    .select("inspection_key", "inspection_id", "inspection_score")
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("inspection_key", "inspection_id")
)

fact_violations = (
    silver
    .filter(
        F.col("violation_code").isNotNull() &
        (F.trim(F.col("violation_code")) != ""))
    
    # Resolve violation_code_key from dim_violation
    .join(dim_viol_lkp,
          on=["violation_id"],
          how="left")
    # Resolve inspection_key from fact_inspections
    .join(fact_insp_lkp,
          on="inspection_id",
          how="left")
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "inspection_key",
        "violation_code_key",
        F.col("violation_points"),
        F.col("is_critical_flag").alias("is_critical"),
        F.col("inspector_comments").alias("violation_comment"),
        "_date_to_warehouse"
    )
)

fact_violations.createOrReplaceTempView("staging_fact_violations")

# Append Only
spark.sql(f""" 
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{FACT_VIOLATIONS} target
    USING staging_fact_violations source
    ON target.inspection_key = source.inspection_key
    AND target.violation_code_key = source.violation_code_key

    -- Safe Append 
    WHEN NOT MATCHED THEN INSERT ( 
        inspection_key, violation_code_key, violation_points, is_critical, violation_comment, _date_to_warehouse
    )
    Values (
        source.inspection_key, source.violation_code_key, source.violation_points, source.is_critical, source.violation_comment, source._date_to_warehouse
    )
""")

get_merge_stats(FACT_VIOLATIONS).display()

timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-11T23:33:25.000Z,null,MERGE,2,240213,2,Data Inserted


In [0]:
# %sql

# select Inspection_Key, violation_code_key, count(1)
# from final_project.gold_zone.fact_violations
# group by 1, 2
# having count(1) > 1
# order by 3 desc

Inspection_Key,violation_code_key,count(1)
